# Evaluating cell segmentations without ground truth

Compares the cell segmentations of one `.zarr` (written by `macroseg.baseline`) with four measures.
Each looks at one kind of error and has a blind spot, so read them together:

| Measure | Error it sees | Blind spot |
|---|---|---|
| Nuclei per cell | merged cells (2+), fragments or false cells (0), missed cells (nuclei in no cell) | not outlines; trivially perfect for grown nuclei |
| Border ratio | outline not on the membrane (ratio near 1) | favours segmentations that used the membrane channel |
| Marker separation (Ruth) | signals of neighbouring cells mixed (many "half positive" cells) | also depends on biology and staining: compare candidates, not absolute values |
| Shape | abnormal objects (huge, ragged) | macrophages are genuinely irregular |

Some cells without a nucleus are expected: the section is thinner than a cell.
Runs on Mewtwo (CPU only, no GPU).

In [ ]:
from pathlib import Path

from spatialdata import read_zarr

from macroseg.config import load_config
from macroseg.evaluate import evaluate_candidates, plot_comparison

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_mouse_neuroblastoma.yaml")
sdata = read_zarr(cfg.zarr)

NUCLEI = "nuclei"          # reference nuclear segmentation
MEMBRANE = "NaK_ATPase"    # channel for the border ratio
MARKER = "F480"            # population marker for the separation measure
CANDIDATES = [step.labels for step in cfg.segmentation if step.labels != NUCLEI]
CANDIDATES

## Summary
One column per candidate. Percentages are of cells, except `nuclei_in_no_cell_%` (of nuclei).

In [ ]:
summary, per_cell = evaluate_candidates(
    sdata, cfg.image.element, CANDIDATES, nuclei=NUCLEI, membrane=MEMBRANE, marker=MARKER
)
summary.T.round(2)

## Distributions

In [ ]:
plot_comparison(per_cell, marker=MARKER, pixel_size_um=sdata.attrs["pixel_size_um"]);

## Cells to look at in napari
For example the largest cells with 2+ nuclei of one candidate (`cell_ID` is the value in its labels element).

In [ ]:
name = CANDIDATES[-1]
per_cell[name].query("n_nuclei >= 2").sort_values("area", ascending=False).head(10)